In [1]:
from minicamels import MiniCamels

ds = MiniCamels()
print("MiniCamels loaded successfully")

MiniCamels loaded successfully


In [3]:
import minicamels
dir(minicamels)

['MiniCamels',
 '__all__',
 '__builtins__',
 '__cached__',
 '__doc__',
 '__file__',
 '__loader__',
 '__name__',
 '__package__',
 '__path__',
 '__spec__',
 '__version__',
 '_constants',
 '_version',
 'dataset',
 'io']

In [11]:
from minicamels import MiniCamels
type(MiniCamels)


type

In [12]:
import minicamels

for name in dir(minicamels):
    obj = getattr(minicamels, name)
    print(f"{name:20} → {type(obj)}")

MiniCamels           → <class 'type'>
__all__              → <class 'list'>
__builtins__         → <class 'dict'>
__cached__           → <class 'str'>
__doc__              → <class 'str'>
__file__             → <class 'str'>
__loader__           → <class '_frozen_importlib_external.SourceFileLoader'>
__name__             → <class 'str'>
__package__          → <class 'str'>
__path__             → <class 'list'>
__spec__             → <class '_frozen_importlib.ModuleSpec'>
__version__          → <class 'str'>
_constants           → <class 'module'>
_version             → <class 'module'>
dataset              → <class 'module'>
io                   → <class 'module'>


In [13]:
from minicamels import MiniCamels
ds = MiniCamels()
print(ds)

MiniCamels(basins=50, period=1980-10-01–2010-09-30, forcing=Daymet, source=remote (https://raw.githubusercontent.com/BennettHydroLab/minicamels/main/data/))


In [17]:

help(ds.__class__)

Help on class MiniCamels in module minicamels.dataset:

class MiniCamels(builtins.object)
 |  MiniCamels(local_data_dir: 'str | Path | None' = None, base_url: 'str' = 'https://raw.githubusercontent.com/BennettHydroLab/minicamels/main/data/')
 |  
 |  Interface to the minicamels dataset.
 |  
 |  By default, data is read from a local ``data/`` directory when the
 |  package is used from within a git clone (e.g. after ``pip install -e .``).
 |  When no local data is found, files are fetched transparently from GitHub
 |  raw URLs — no download or configuration needed.
 |  
 |  Parameters
 |  ----------
 |  local_data_dir
 |      Explicit path to a local ``data/`` directory. Pass ``None`` (default)
 |      to auto-detect from the repository layout, then fall back to remote.
 |  base_url
 |      Remote base URL. Override only if using a fork or mirror.
 |  
 |  Examples
 |  --------
 |  >>> ds = MiniCamels()
 |  >>> ds.basins().head()
 |  >>> ts = ds.load_basin("01013500")
 |  >>> ts["prcp"

In [21]:
from minicamels import MiniCamels

def test_data():
    ds = MiniCamels()

    basins = ds.basins()
    print("Basins table:")
    print(basins.head())
    print("Number of basins:", len(basins))

    basin_id = basins.iloc[0]["basin_id"]
    print("Sample basin:", basin_id)

    data = ds.open_basin(basin_id)

    print(data)
    print("Variables:")
    print(list(data.data_vars))

if __name__ == "__main__":
    test_data()

Basins table:
   basin_id                               basin_name
0  01013500         Fish River near Fort Kent, Maine
1  01580000                  DEER CREEK AT ROCKS, MD
2  02016000  COWPASTURE RIVER NEAR CLIFTON FORGE, VA
3  02064000            FALLING RIVER NEAR NARUNA, VA
4  02221525          MURDER CREEK BELOW EATONTON, GA
Number of basins: 50
Sample basin: 01013500
<xarray.Dataset> Size: 351kB
Dimensions:  (time: 10957)
Coordinates:
  * time     (time) datetime64[ns] 88kB 1980-10-01 1980-10-02 ... 2010-09-30
Data variables:
    prcp     (time) float32 44kB ...
    tmax     (time) float32 44kB ...
    tmin     (time) float32 44kB ...
    srad     (time) float32 44kB ...
    vp       (time) float32 44kB ...
    qobs     (time) float32 44kB ...
Attributes:
    Conventions:  CF-1.8
    basin_id:     01013500
    basin_name:   Fish River near Fort Kent, Maine
    area_km2:     2252.7
    lat:          47.23739
    lon:          -68.58264
    huc02:        01
    forcing:      Daymet

In [23]:
from minicamels import MiniCamels
import numpy as np
from sklearn.preprocessing import StandardScaler

def create_sequences(data, seq_len=30):
    X_vars = ["prcp", "tmax", "tmin", "srad", "vp"]
    y_var = "qobs"

    X = np.stack([data[var].values for var in X_vars], axis=1)
    y = data[y_var].values

    X_seq, y_seq = [], []

    for i in range(len(X) - seq_len):
        X_seq.append(X[i:i+seq_len])
        y_seq.append(y[i+seq_len])

    return np.array(X_seq), np.array(y_seq)


def normalize_data(X, y):
    # reshape X for scaling
    n_samples, seq_len, n_features = X.shape

    X_reshaped = X.reshape(-1, n_features)

    scaler_X = StandardScaler()
    X_scaled = scaler_X.fit_transform(X_reshaped)

    X_scaled = X_scaled.reshape(n_samples, seq_len, n_features)

    scaler_y = StandardScaler()
    y_scaled = scaler_y.fit_transform(y.reshape(-1, 1)).flatten()

    return X_scaled, y_scaled, scaler_X, scaler_y


def train_test_split(X, y, split_ratio=0.8):
    split = int(len(X) * split_ratio)

    X_train = X[:split]
    y_train = y[:split]

    X_test = X[split:]
    y_test = y[split:]

    return X_train, X_test, y_train, y_test


def test_pipeline():
    ds = MiniCamels()
    basins = ds.basins()

    basin_id = basins.iloc[0]["basin_id"]
    data = ds.open_basin(basin_id)

    X, y = create_sequences(data, seq_len=30)

    X, y, scaler_X, scaler_y = normalize_data(X, y)

    X_train, X_test, y_train, y_test = train_test_split(X, y)

    print("Train shape:", X_train.shape, y_train.shape)
    print("Test shape:", X_test.shape, y_test.shape)

    print("Sample normalized input:")
    print(X_train[0])

    print("Sample normalized target:")
    print(y_train[0])


if __name__ == "__main__":
    test_pipeline()

Train shape: (8741, 30, 5) (8741,)
Test shape: (2186, 30, 5) (2186,)
Sample normalized input:
[[-6.41116931e-04  7.10876882e-02  3.97360146e-01 -7.50899911e-01
   9.61268097e-02]
 [ 2.19107285e-01  5.47942519e-01  6.58930540e-01 -6.56472802e-01
   4.90269363e-01]
 [ 9.47746933e-01  5.51248252e-01  8.73702407e-01 -9.38789248e-01
   8.40153039e-01]
 [ 2.34526992e+00  1.57863721e-01  5.26891828e-01 -9.77180481e-01
   2.70489812e-01]
 [ 1.03641725e+00 -1.33042574e-01  3.89838964e-01 -1.15604067e+00
   8.38571712e-02]
 [-5.98202646e-01 -1.22298874e-01  4.80928987e-01 -1.10200357e+00
   2.12067142e-01]
 [-5.98202646e-01  1.66686345e-03  5.21877646e-01 -8.99088562e-01
   2.59967029e-01]
 [-1.93402901e-01  2.47119054e-01  4.85943109e-01 -5.71695089e-01
   2.25620538e-01]
 [ 8.61016884e-02  5.12531921e-02  3.17134082e-01 -7.69440770e-01
  -2.78530847e-02]
 [-5.98202646e-01 -1.15687370e-01  1.83423996e-01 -4.51696754e-01
  -1.53642744e-01]
 [ 1.45663798e+00 -1.08249418e-01 -1.81771725e-01 -4.361

In [24]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size=5, hidden_size=64, num_layers=2):
        super(LSTMModel, self).__init__()

        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True
        )

        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        # x shape: (batch, seq_len, features)

        out, _ = self.lstm(x)

        # take last timestep
        out = out[:, -1, :]

        out = self.fc(out)

        return out

In [25]:
import torch
from model import LSTMModel

def test_model():
    model = LSTMModel()

    # fake input
    x = torch.randn(32, 30, 5)

    y = model(x)

    print("Input shape:", x.shape)
    print("Output shape:", y.shape)

if __name__ == "__main__":
    test_model()

ImportError: cannot import name 'LSTMModel' from 'model' (/Users/khrahaman@arizona.edu/streamflow_project/model.py)